<div dir="rtl">

# מתמונות לנתונים: החממה האקולוגית עין שמר

המחברת מריצה **אותו קוד מדידה על כל התמונות** ושומרת טבלה (`results.csv`) ותמונות מסומנות לנספח.

**לפני הכול:** לוחצים למעלה על **Copy to Drive** (בעברית: "העתקה ל-Drive"). העותק נפתח בחלון חדש, ועובדים רק עליו.

**איך עובדים:** מריצים את התאים לפי הסדר, בכפתור ▶ שמשמאל לכל תא. תא שרץ מקבל ✓.
שגיאה? מעתיקים את הודעת השגיאה לדאטה-בוט, או לוחצים על "הסבר את השגיאה" של Gemini.

</div>

<div dir="rtl">

## 1. התמונות
מעלים את כל תמונות הניסוי **לתיקייה אחת** ב-"האחסון שלי" בדרייב, וכותבים כאן את שם התיקייה.
התמונות צריכות להיות JPG או PNG. תמונות HEIC מאייפון לא נקראות: מורידים אותן מגוגל תמונות כ-JPG.

</div>

In [ ]:
#@title 1. חיבור לתיקיית התמונות { display-mode: "form" }
FOLDER = "ניסוי" #@param {type:"string"}

import os, glob, cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PATH = os.path.join('/content/drive/MyDrive', FOLDER)
except ImportError:  # הרצה מחוץ ל-Colab (בדיקה)
    PATH = FOLDER

def read_image(f):
    # קריאה שעובדת גם עם שמות בעברית
    return cv2.imdecode(np.fromfile(f, dtype=np.uint8), cv2.IMREAD_COLOR)

def write_image(f, img):
    cv2.imencode('.jpg', img)[1].tofile(f)

files = sorted(f for f in glob.glob(os.path.join(PATH, '*'))
               if f.lower().endswith(('.jpg', '.jpeg', '.png')))
heic = glob.glob(os.path.join(PATH, '*.[hH][eE][iI][cC]'))
if not os.path.isdir(PATH):
    print('❌ לא מצאתי את התיקייה "' + FOLDER + '" ב"האחסון שלי". בדקו את השם (רווחים ואותיות).')
else:
    print(f'✓ נמצאו {len(files)} תמונות בתיקייה "{FOLDER}"')
    if heic:
        print(f'⚠ {len(heic)} תמונות HEIC לא ייקראו. הורידו אותן כ-JPG.')

<div dir="rtl">

## 2. מה מודדים
בוחרים אחת משתי הדוגמאות המוכנות:
- **אחוז ירוק:** שטח הצמח, אחוז ירוק ואחוז צהוב. למשל עלים או מורינגה.
- **ספירה וגודל:** מספר האובייקטים, הגודל שלהם ואחוז הכיסוי. למשל עדשות מים.

**קנה מידה:** כמה פיקסלים יש בסנטימטר אחד. אם משאירים 0, השטחים יוצאים בפיקסלים.
**גודל מינימלי:** כתם שקטן מזה לא נספר, וכך מסננים לכלוך.

</div>

In [ ]:
#@title 2. בחירת מדידה { display-mode: "form" }
MODE = "אחוז ירוק" #@param ["אחוז ירוק", "ספירה וגודל"]
PX_PER_CM = 0 #@param {type:"number"}
MIN_AREA_PX = 150 #@param {type:"integer"}

# טווחי צבע ב-HSV (גוון 0–179 ב-OpenCV)
GREEN = ((35, 40, 40), (90, 255, 255))
YELLOW = ((18, 60, 60), (34, 255, 255))

def clean(mask):
    # מסיר נקודות קטנות ומחבר חורים קטנים
    k = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)

def objects(mask):
    # רק כתמים גדולים מהגודל המינימלי
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = [c for c in cnts if cv2.contourArea(c) >= MIN_AREA_PX]
    # מספור מלמעלה למטה ומימין לשמאל, כדי שיהיה קל לבדוק ביד
    h = max(1, (cnts and int(np.median([cv2.boundingRect(c)[3] for c in cnts]))) or 1)
    return sorted(cnts, key=lambda c: (cv2.boundingRect(c)[1] // h, -cv2.boundingRect(c)[0]))

def area(px):
    return round(px / PX_PER_CM ** 2, 2) if PX_PER_CM > 0 else int(px)

UNIT = 'סמ"ר' if PX_PER_CM > 0 else 'פיקסלים'

def measure(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    green = cv2.inRange(hsv, *GREEN)
    yellow = cv2.inRange(hsv, *YELLOW)
    cnts = objects(clean(green | yellow))
    plant = np.zeros(green.shape, np.uint8)
    cv2.drawContours(plant, cnts, -1, 255, -1)
    total = cv2.countNonZero(plant)
    g = cv2.countNonZero(green & plant)
    y = cv2.countNonZero(yellow & plant & ~green)
    marked = img.copy()
    marked[(green & plant) > 0] = (0, 200, 0)
    marked[(yellow & plant & ~green) > 0] = (0, 220, 255)
    cv2.drawContours(marked, cnts, -1, (255, 0, 255), max(2, img.shape[1] // 400))
    res = {f'שטח צמח ({UNIT})': area(total),
           'אחוז ירוק': round(100 * g / total, 1) if total else 0,
           'אחוז צהוב': round(100 * y / total, 1) if total else 0}
    if MODE == 'ספירה וגודל':
        areas = [cv2.contourArea(c) for c in cnts]
        med = float(np.median(areas)) if areas else 0
        fs = max(0.5, img.shape[1] / 1500)
        for i, c in enumerate(cnts, 1):
            m = cv2.moments(c)
            if m['m00']:
                cx, cy = int(m['m10'] / m['m00']), int(m['m01'] / m['m00'])
                cv2.putText(marked, str(i), (cx, cy), cv2.FONT_HERSHEY_SIMPLEX, fs, (0, 0, 0), max(1, int(fs * 2)))
        res = {'מספר': len(cnts),
               # כתמים שנוגעים זה בזה נספרים כאחד. התיקון: שטח כולל חלקי שטח של אובייקט בודד טיפוסי
               'ספירה מתוקנת': round(sum(areas) / med, 1) if med else 0,
               f'גודל ממוצע ({UNIT})': area(np.mean(areas)) if areas else 0,
               f'גודל חציוני ({UNIT})': area(med),
               'אחוז כיסוי': round(100 * total / plant.size, 1),
               'אחוז ירוק': res['אחוז ירוק']}
    return res, marked

print('✓ נבחרה מדידה:', MODE, '| יחידות:', UNIT)

<div dir="rtl">

## 3. קוד מדאטה-בוט (לא חובה)
אם דאטה-בוט כתב לכם קוד מדידה מותאם לניסוי, מדביקים אותו כאן ומריצים. הוא **מחליף** את הדוגמה מתא 2.
אם לא, מדלגים לתא 4.

</div>

In [ ]:
# הדביקו כאן את הקוד מדאטה-בוט (פונקציה measure(img) שמחזירה results, marked)

<div dir="rtl">

## 4. כיול: בדיקה על תמונה אחת
משמאל התמונה המקורית, ומימין מה שהקוד מדד. **נראה נכון?** משהו נספר פעמיים, או שמשהו חסר?
אם לא, משנים את ההגדרות בתא 2 (או מבקשים מדאטה-בוט לתקן את הקוד), מריצים שוב את תא 2 ואת התא הזה.

</div>

In [ ]:
#@title 4. בדיקה על תמונה אחת { display-mode: "form" }
IMAGE_NUMBER = 1 #@param {type:"integer"}

f = files[IMAGE_NUMBER - 1]
img = read_image(f)
res, marked = measure(img)
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax[0].set_title('original')
ax[1].imshow(cv2.cvtColor(marked, cv2.COLOR_BGR2RGB)); ax[1].set_title('measured')
for a in ax: a.axis('off')
plt.show()
print(os.path.basename(f))
for k, v in res.items():
    print(f'  {k}: {v}')

<div dir="rtl">

## 5. הרצה על כל התמונות
אותו קוד, על כל התמונות בתיקייה. התמונות המסומנות נשמרות בתת-תיקייה `marked`, לנספח.

</div>

In [ ]:
#@title 5. הרצה על כל התמונות { display-mode: "form" }
MARKED = os.path.join(PATH, 'marked')
os.makedirs(MARKED, exist_ok=True)
rows = []
for i, f in enumerate(files, 1):
    name = os.path.basename(f)
    img = read_image(f)
    if img is None:
        rows.append({'קובץ': name, 'הערות': 'התמונה לא נקראה'})
        continue
    res, marked = measure(img)
    write_image(os.path.join(MARKED, os.path.splitext(name)[0] + '_marked.jpg'), marked)
    rows.append({'קובץ': name, **res})
    print(f'\r{i}/{len(files)}', end='')
df = pd.DataFrame(rows)
print(f'\n✓ נמדדו {len(df)} תמונות. התמונות המסומנות בתיקייה marked')
try:
    display(df)
except NameError:
    print(df)

<div dir="rtl">

## 6. טיפולים ושמירה
**מאיפה הטיפול?** דאטה-בוט יכול לקרוא את הפתקים בתמונות ולתת לכם טבלה "קובץ,טיפול". מדביקים אותה בין שלושת המירכאות.
אם אין, משאירים ריק, ומוסיפים את הטיפול בגיליון.

הקובץ `results.csv` נשמר בתיקיית התמונות. בדרייב: לחיצה ימנית ← פתיחה באמצעות ← Google Sheets.

</div>

In [ ]:
TREATMENTS = """
קובץ,טיפול
"""

import io
out = df.copy()
t = pd.read_csv(io.StringIO(TREATMENTS.strip()))
if len(t):
    t.columns = ['קובץ', 'טיפול']
    out = out.merge(t, on='קובץ', how='left')
    out.insert(1, 'טיפול', out.pop('טיפול'))
    missing = out['טיפול'].isna().sum()
    if missing:
        print(f'⚠ ל-{missing} תמונות אין טיפול. בדקו ששמות הקבצים זהים בדיוק.')
csv = os.path.join(PATH, 'results.csv')
out.to_csv(csv, index=False, encoding='utf-8-sig')
print('✓ נשמר:', csv)

<div dir="rtl">

## מה כותבים בעבודה
- **בשיטות:** איך צילמתם (מרחק, תאורה, קנה מידה), ושהמדידה נעשתה ב-Python עם OpenCV ב-Google Colab: מה נמדד, באיזה סף צבע ומה הגודל המינימלי.
- **בנספח:** הקוד (תא 2 או 3), ו-2–3 תמונות מסומנות.
- **במגבלות:** כתמים שנוגעים זה בזה נספרים כאחד (לכן יש ספירה מתוקנת), צל או רקע בצבע הצמח משבשים, ובלי קנה מידה המספרים הם בפיקסלים.

</div>